# Detecting Renal Cysts from CT Reports Using DistilBERT base model (uncased)


In this project, I used the DistilBERT base model to classify CT reports based on the identification of renal cysts. For efficiency, I froze the model's pretrained weights and optimized only the head layer.


## 1. Load Dataset



*   Load dataset from Google Drive
*   Drop unnecessary features
*   Check Data types


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

#load dataset
data = pd.read_csv("path_to_your_dataset.csv")

#Drop unnessary features

data=data.drop(['MRN','PRODUCT','PATIENT_WEIGHT','NATIONALITY','EPISODE_DATE','PATIENT_HEIGHT','Unnamed: 10','PATIENT_BMI'], axis=1)

data.head()

In [ ]:
#check data type
print(data.dtypes)

PATIENT_AGE       object
PATIENT_GENDER    object
CT_REPORT         object
dtype: object


##  2. Preprocessing

In this step:


*   We convert datatypes for all columns
*   We perfromed cleaning CT_REPORT
*   We auto labeled data using regular expression since data is too large and was not labeld manually.




### Convert data types and remove Y from age column

In [ ]:
# remove Y from age column
data["PATIENT_AGE"] = data["PATIENT_AGE"].str.replace("Y", "", regex=False)

# convert data types
data["PATIENT_AGE"] = data["PATIENT_AGE"].astype(int)
data["PATIENT_GENDER"] = data["PATIENT_GENDER"].astype(str)
data["CT_REPORT"] = data["CT_REPORT"].astype(str)

# check data types
print(data.dtypes)

PATIENT_AGE        int64
PATIENT_GENDER    object
CT_REPORT         object
dtype: object


### Perfrom cleaning for CT REPORT

In [ ]:
import re


def clean_ct_report(report):
    """
    Cleans escape characters and unwanted spaces from a CT report.
    """
    # remove newlines, tabs, and carriage returns and other noisy charcaters observed in the reports such as \T\
    report = report.replace("\n", " ").replace("\t", " ").replace("\r", " ").replace("\T\\", " ")

    # this extra step to remove Unicode escape sequences, if any
    report = re.sub(r"\\u[0-9A-Fa-f]{4}", "", report)  # Matches \uXXXX
    report = re.sub(r"\\x[0-9A-Fa-f]{2}", "", report)  # Matches \xXX

    # replace multiple spaces with a single space, if any
    report = re.sub(r"\s+", " ", report)

    # strip leading and trailing spaces, if any
    report = report.strip()

    return report

# apply cleaning
data["CT_REPORT"] = data["CT_REPORT"].apply(clean_ct_report)

print(data)

### Auto labeling data
Since data is large and is not labeled, I used rule-based approach forlabeling data using regular expression where (1: renal cyst, 0: no renal cyst)






In [ ]:
# Define a labeling function
def contains_renal_cyst(text):
    return 1 if re.search(r"\brenal cyst\b", text, re.IGNORECASE) else 0

# Apply the function to create label
data["label"] = data["CT_REPORT"].apply(contains_renal_cyst)

print(data)


In [ ]:
# Filter records with label=1 to check labeling
data_filtered = data[data['label'] == 1]
print(data_filtered)

## 3. Prparing For Model Training



*   Split dataset into train-test, with 40% for testing
*   Define model and toeknizer
*   Tokenize data




### Split data set for train-test

In [ ]:
from sklearn.model_selection import train_test_split

# split the dataset for train-test
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data["CT_REPORT"].tolist(),
    data["label"].tolist(),
    test_size=0.4,
    stratify=data["label"],
    random_state=42,
)


### Define Model and Tokenizer



In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader
import torch

# load the pre-trained model and tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2, id2label={0: "no renal cyst", 1: "renal cyst"},
    label2id={"no renal cyst": 0, "renal cyst": 1})


# freeze weights of the pre-trained model
for param in model.base_model.parameters():
    param.requires_grad = False

print(model)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


### Tokenize data

In [ ]:
from torch.utils.data import Dataset

# Tokenize the data
def tokenize_data(texts, labels):
    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    tokens["labels"] = torch.tensor(labels)  # Add labels to the tokens
    return tokens

# Custom Dataset Class
class CTReportDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
        self.encodings = tokenize_data(texts, labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

# Prepare datasets
train_dataset = CTReportDataset(train_texts, train_labels)
test_dataset = CTReportDataset(test_texts, test_labels)


## 4. Training Setup and Execution

In [ ]:
from transformers import Trainer,TrainingArguments,DataCollatorWithPadding
import numpy as np
from sklearn.metrics import accuracy_score


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}


# training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.120775,0.975054
2,0.143700,0.118554,0.975054


TrainOutput(global_step=776, training_loss=0.13664406353665381, metrics={'train_runtime': 364.3396, 'train_samples_per_second': 33.996, 'train_steps_per_second': 2.13, 'total_flos': 1640741199753216.0, 'train_loss': 0.13664406353665381, 'epoch': 2.0})

## 5. Model Evaluation

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.11855369806289673, 'eval_accuracy': 0.9750544926132235, 'eval_runtime': 67.6796, 'eval_samples_per_second': 61.008, 'eval_steps_per_second': 3.827, 'epoch': 2.0}
